# Ways to Measure an Annual Return
### (+ annualised risk and a comparison chart)

Take daily stock prices and turn them into an **annual return** — but in **four
different flavours**, add **annualised risk**, and draw the important distinction
between two ways of averaging.

**The four return methods** (all *annualised*, i.e. expressed "per year"):

| | Method | Formula | Note |
|---|---|---|---|
| **M1** | Point-to-point (real, compounded) | `(last_price / first_price) - 1` | The actual return of buying in Jan and selling in Dec. |
| **M2** | Mean **simple** daily return × 252 | mean of `(today/yesterday - 1)`, ×252 | A common shortcut. Slightly **overstates** the truth (ignores compounding). |
| **M3** | Mean **log** daily return × 252 | mean of `ln(today/yesterday)`, ×252 | A continuously-compounded rate. Cleaner maths than M2. |
| **M4** | M3 converted back to a normal % | `exp(M3) - 1` | Puts M3 on the same scale as M1/M2 for a like-for-like comparison. |

**The averaging distinction (global vs year-by-year).** When we average daily
returns for M2/M3, over *what window*?

- **Global** — average **all** daily returns across the whole 5-year sample, then
  ×252 → **one number per stock**. *"What's this stock's typical annual return?"*
- **Year-by-year** — average only the daily returns **within one calendar year**,
  then ×252 → **one number per stock per year**. *"What did it do in 2023 specifically?"*

We compute **both** and place them side by side.

**Risk.** Annualised risk = (standard deviation of daily returns) × √252.
Volatility scales with the **square root** of time, which is why risk uses **√252**
while returns use plain **252**.

> **Note on running this notebook:** the first cell downloads live prices from Yahoo
> Finance via `yfinance`, so you need an internet connection. Each run pulls the most
> recent 5 years, so your exact numbers will differ from any earlier run.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## 1 · Setup and download

Pick the tickers and the date window (the last 5 whole years), then download the
**adjusted close** price for each. Adjusted close accounts for dividends and splits,
so returns computed from it are apples-to-apples over time.

In [ ]:
tickers = ['SPY', 'NVDA', 'AIR.PA', 'MSFT', 'BA', 'JPM',
           'XOM', 'DGE.L', 'ANF', 'DPZ', 'SAP.DE']

end_date = pd.Timestamp.today().normalize()
start_date = pd.Timestamp(year=end_date.year - 5, month=1, day=1)

print(f"Downloading prices from {start_date.date()} to {end_date.date()} ...")
adj_close_df = yf.download(tickers,
                           start=start_date,
                           end=end_date,
                           auto_adjust=False)['Adj Close']

adj_close_df.tail()

## 2 · Reshape to long (tidy) format and compute daily returns

The download gives a **wide** table (one column per ticker). We reshape it to
**long/tidy** format — one row per (date, ticker) — because it makes grouped
calculations and seaborn plotting much easier.

- `.melt()` stacks the ticker columns into a single `Ticker` column plus a `Adj_Close` value column.
- `.groupby('Ticker')` keeps each stock's calculation separate (so we never compute a
  return across two different companies).
- **Simple return** = `(today / yesterday) - 1`, via `.pct_change()`.
- **Log return** = `ln(today / yesterday)`.

In [ ]:
long_df = (adj_close_df
           .reset_index()
           .melt(id_vars='Date', var_name='Ticker', value_name='Adj_Close')
           .dropna()
           .sort_values(['Ticker', 'Date'])
           .reset_index(drop=True))

long_df['Year'] = long_df['Date'].dt.year

# Simple daily return: (today / yesterday) - 1
long_df['Simple_Return'] = long_df.groupby('Ticker')['Adj_Close'].pct_change()
# Log daily return: ln(today / yesterday)
long_df['Log_Return'] = (long_df.groupby('Ticker')['Adj_Close']
                         .transform(lambda p: np.log(p / p.shift(1))))

TRADING_DAYS = 252

long_df.head()

## 3 · Global metrics — one number per stock (all years at once)

Group by **ticker only**, so every average spans the entire sample. Everything
ending in `_global` uses the whole 5 years.

In [ ]:
global_metrics = (long_df.groupby('Ticker')
                  .agg(mean_simple=('Simple_Return', 'mean'),
                       mean_log=('Log_Return', 'mean'),
                       std_log=('Log_Return', 'std'),      # risk from LOG returns
                       first_px=('Adj_Close', 'first'),
                       last_px=('Adj_Close', 'last'),
                       n_days=('Simple_Return', 'count'))
                  .reset_index())

# Annualise
global_metrics['M2_global'] = global_metrics['mean_simple'] * TRADING_DAYS
global_metrics['M3_global'] = global_metrics['mean_log'] * TRADING_DAYS
global_metrics['M4_global'] = np.exp(global_metrics['M3_global']) - 1   # M3 -> simple

# Risk uses LOG returns: log returns add across time, which is exactly the
# assumption behind the sqrt(252) scaling (and it matches M3). For daily data
# the number is virtually identical to using simple returns.
global_metrics['Risk_global'] = global_metrics['std_log'] * np.sqrt(TRADING_DAYS)

# A "true" whole-sample M1: the compound annual growth rate (CAGR), annualised.
global_metrics['M1_global_CAGR'] = (
    (global_metrics['last_px'] / global_metrics['first_px'])
    ** (TRADING_DAYS / global_metrics['n_days']) - 1
)

global_metrics.round(4)

## 4 · Year-by-year metrics — one number per stock *per year*

Same calculations, but group by **both** `Year` and `Ticker`, so each average is
taken **within a single calendar year**.

In [ ]:
yearly = (long_df.groupby(['Year', 'Ticker'])
          .agg(mean_simple=('Simple_Return', 'mean'),
               mean_log=('Log_Return', 'mean'),
               std_log=('Log_Return', 'std'),        # risk from LOG returns
               first_px=('Adj_Close', 'first'),
               last_px=('Adj_Close', 'last'),
               Days=('Simple_Return', 'count'))
          .reset_index())

# M1: the real point-to-point return for that year (first vs last price).
yearly['M1_point_to_point'] = yearly['last_px'] / yearly['first_px'] - 1
# M2 / M3 / M4: the shortcuts, but averaged WITHIN the year only.
yearly['M2_yearly'] = yearly['mean_simple'] * TRADING_DAYS
yearly['M3_yearly'] = yearly['mean_log'] * TRADING_DAYS
yearly['M4_yearly'] = np.exp(yearly['M3_yearly']) - 1
# Annualised risk for that single year, from LOG returns.
yearly['Risk_yearly'] = yearly['std_log'] * np.sqrt(TRADING_DAYS)

yearly.head()

## 5 · Combine — put each year next to the stock's global average

Merge the whole-sample columns onto every yearly row, so each line shows
this year beside the stock's long-run average.

In [ ]:
comparison = yearly.merge(
    global_metrics[['Ticker', 'M2_global', 'M3_global', 'Risk_global']],
    on='Ticker', how='left'
)

# Choose and order the columns we actually want to see.
cols = ['Year', 'Ticker', 'Days',
        'M1_point_to_point',                      # real return, this year
        'M2_yearly', 'M3_yearly', 'M4_yearly',    # shortcuts, this year
        'Risk_yearly',                            # risk, this year
        'M2_global', 'M3_global',                 # long-run average shortcuts
        'Risk_global']                            # long-run average risk
comparison = comparison[cols]
comparison.head()

## 6 · Show the tables

We format everything as a percentage per year for readability.

In [ ]:
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 300)
pd.set_option('display.float_format', lambda x: f'{x*100:6.2f}%')

print("=== YEAR-BY-YEAR vs GLOBAL (returns & risk as % per year) ===")
print(comparison.to_string(index=False))

print("\n=== GLOBAL (whole-sample) summary, one row per stock ===")
gshow = global_metrics[['Ticker', 'M1_global_CAGR', 'M2_global',
                        'M3_global', 'M4_global', 'Risk_global']]
print(gshow.to_string(index=False))

## 7 · Compare the four return methods (whole-sample)

We chart the **global** numbers because that's one clean value per stock per method
(charting all 55 year×stock combinations would be unreadable). We reshape to long
format so seaborn can put the four methods side by side for each stock.

In [ ]:
sns.set_style("whitegrid")

# Left-to-right stock order (highest M2 first) so the chart is easy to scan.
ticker_order = (global_metrics.sort_values('M2_global', ascending=False)['Ticker']
                .tolist())

# --- Reshape to LONG format for seaborn: one row per (stock, method) ---
method_names = {
    'M1_global_CAGR': 'M1  point-to-point (CAGR)',
    'M2_global':      'M2  simple mean x252',
    'M3_global':      'M3  log mean x252',
    'M4_global':      'M4  M3 -> simple (exp-1)',
}
returns_long = (global_metrics
                .melt(id_vars='Ticker',
                      value_vars=list(method_names),     # the four method columns
                      var_name='Method',
                      value_name='Return'))
returns_long['Return'] *= 100                            # to percent
returns_long['Method'] = returns_long['Method'].map(method_names)   # nice labels

# --- Draw the grouped bar chart ---
plt.figure(figsize=(14, 6))
ax = sns.barplot(data=returns_long, x='Ticker', y='Return',
                 hue='Method', order=ticker_order)
ax.axhline(0, color='gray', lw=1)
ax.set_title('Four ways to measure annualised return (whole-sample)')
ax.set_xlabel('')
ax.set_ylabel('Annualised return (% per year)')
ax.legend(title='Method', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8 · Risk–return chart, year by year

One small scatter per calendar year: **risk on the x-axis, return on the y-axis**,
each point a stock. Both metrics come from **log** daily returns:

- `Annual_Return = exp(mean(log return) × 252) - 1`  (annualise, then convert back to a normal %, i.e. M4)
- `Annual_Risk   = std(log return) × √252`

In [ ]:
annual_metrics = yearly.assign(
    Annual_Return=np.exp(yearly['mean_log'] * TRADING_DAYS) - 1,
    Annual_Risk=yearly['std_log'] * np.sqrt(TRADING_DAYS),
)

print("--- Annual Risk-Return Metrics (Tidy Format) ---")
print(annual_metrics[['Year', 'Ticker', 'Annual_Return', 'Annual_Risk']]
      .head(10).to_string())

In [ ]:
print("Generating plot...")
g = sns.relplot(
    data=annual_metrics,
    x='Annual_Risk',
    y='Annual_Return',
    hue='Ticker',
    col='Year',
    col_wrap=3,
    s=150,
    alpha=0.8,
    height=5,
    aspect=1.2,
    kind='scatter',
    legend=False,
    facet_kws={'sharey': False}   # let each year's panel auto-scale its own y-axis
)

for year, ax in g.axes_dict.items():
    ax.set_title(f"Year: {year}")
    year_data = annual_metrics[annual_metrics['Year'] == year]

    # Direct text label next to each point
    for idx, row in year_data.iterrows():
        ax.annotate(
            text=row['Ticker'],
            xy=(row['Annual_Risk'], row['Annual_Return']),
            color='black',
            size=9,
            weight='normal',
            xytext=(7, 0),
            textcoords='offset points'
        )

    # Percent-format the axes and add reference lines
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.axhline(0, ls='--', color='gray', zorder=0)

    if not year_data.empty:
        median_risk = year_data['Annual_Risk'].median()
        ax.axvline(median_risk, ls=':', color='gray', zorder=0, label='Median Risk')

title_str = (f"Annual Risk vs. Return by Year\n"
             f"Data from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
g.fig.suptitle(title_str, fontsize=18)
g.fig.tight_layout(rect=[0, 0.02, 1, 0.95])

g.savefig('risk_return_by_year.png', dpi=150, bbox_inches='tight')
print("Saved chart to 'risk_return_by_year.png'")
plt.show()

## 9 · Save the full table for Excel

In [ ]:
comparison.to_csv('three_annual_returns.csv', index=False)
print("Saved table to 'three_annual_returns.csv'")